# xgrid-gpu on an NVIDIA GPU (Colab)

Runs the CUDA gate (`tests/run_gpu.sh cuda`) and the full-resolution NCSX and W7-X
benchmarks (`201 x 201 x 36`, stellarator symmetry, end to end) on the Colab GPU.

1. **Runtime > Change runtime type**: pick a GPU (T4 on the free tier; L4 or A100 on paid).
2. **Runtime > Run all**. Building takes about a minute and the benchmark a few minutes.
3. Copy the output of the last cell.

From a terminal instead, with the Colab CLI: see "On a Google Colab GPU" in the README.

Consumer and inference cards (T4, L4) run native `fp64` at a small fraction of their `fp32`
rate, so expect `ds` to beat `fp64` there; on an A100 or H100 `fp64` should be fastest.

In [ ]:
!nvidia-smi

In [ ]:
%%bash
# Clone, fetch Zig 0.17.1 (checksum-verified) and build.
set -e
cd /content
[ -d xgrid-gpu ] || git clone -q https://github.com/Lulzx/xgrid-gpu
cd xgrid-gpu && git log --oneline -1
if [ ! -x /content/zig/zig ]; then
  curl -sSLo zig.tar.xz https://ziglang.org/download/0.17.1/zig-x86_64-linux-0.17.1.tar.xz
  echo "3581863e3cbcfd0fce0efc682ce7e9e93da96668a977791d356a26d2a7e9b17a  zig.tar.xz" | sha256sum -c -
  mkdir -p /content/zig && tar -xJf zig.tar.xz -C /content/zig --strip-components=1 && rm zig.tar.xz
fi
/content/zig/zig build --release=fast
zig-out/bin/xgrid-gpu --version

## CUDA gate

Every reference case on the CPU backend and on CUDA in `fp64`, `ds` and `fp32`, compared
point by point. `fp64` must agree with the CPU to 1e-13 and `ds` to 1e-10; `fp32` is reported only.

In [ ]:
%%bash
cd /content/xgrid-gpu && tests/run_gpu.sh cuda

## Benchmark

NCSX and W7-X at `201 x 201 x 36` with symmetry, the timing cases in
`reference/legacy-d05a4a1/cases/ARGS.txt`. Each GPU run is done twice: the first includes the
driver's one-off PTX compilation. Each GPU output is compared with the CPU output.

In [ ]:
%%bash
cd /content/xgrid-gpu
bin=$PWD/zig-out/bin; ref=$PWD/reference/legacy-d05a4a1/cases
work=/content/bench; rm -rf $work; mkdir -p $work
for c in ncsx_full_S_Y:ncsx_S_Y w7x_full_S_Y:w7x_S_Y; do
  name=${c%%:*}; coildir=${c#*:}
  set -- $(grep "^$name " $ref/ARGS.txt)
  ext=$2 mode=$3 rmin=$5 rmax=$6 zmin=$7 zmax=$8 kp=$9 ir=${10} jz=${11}
  d=$work/$name; mkdir -p $d; cp $ref/$coildir/coils.$ext $d/; cd $d
  run() { "$bin/xgrid-gpu" --coils coils.$ext --mode $mode --sym --rmin $rmin --rmax $rmax \
            --zmin $zmin --zmax $zmax --kp $kp --ir $ir --jz $jz --backend $1 --precision $2 \
            --out $3 --extcur $3.extcur 2>&1 | grep -E 'backend|points|time|gpu|error|warning'; }
  echo "===== $name"
  echo "--- cpu fp64 ($(nproc) threads)"; run cpu fp64 cpu.nc
  for p in fp64 ds fp32; do
    for pass in 1 2; do echo "--- cuda $p, run $pass"; run cuda $p $p.nc; done
    case $p in fp64) tol=1e-13;; ds) tol=1e-10;; *) tol=1e300;; esac
    "$bin/compare_mgrid" cpu.nc $p.nc $tol | grep -E 'worst eps|non-finite'
    rm -f $p.nc
  done
  rm -f cpu.nc
done

In [ ]:
%%bash
echo "GPU:    $(nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader)"
echo "CPU:    $(grep -m1 'model name' /proc/cpuinfo | cut -d: -f2) x $(nproc)"
echo "commit: $(cd /content/xgrid-gpu && git rev-parse --short HEAD)"